# 🎓 Lab 3 (Capstone): The Course Assistant — Eyes + Hands + a Plan

The finale. You'll combine everything from this week into one agent:

- 👀 **Eyes:** it searches the course notes (Lab 1's retriever becomes a *tool*)
- 🛠️ **Hands:** it calculates, and it can **save notes to a real file** — a true action with a real side effect
- 🔁 **A plan:** the Think→Act→Observe loop decides what to do, in what order

One agent. Your first complete AI system. Let's go. 🚀

🔑 Needs your OpenRouter API key.

In [20]:
!pip install openai sentence-transformers

### 🛠️ Setup

In [21]:
!pip install openai sentence-transformers -q

import json
from openai import OpenAI
from getpass import getpass
from sentence_transformers import SentenceTransformer, util

api_key = getpass("🔑 Paste your OpenRouter API key: ")
client = OpenAI(api_key=api_key)
MODEL = "gpt-5.4-nano"

def chat(messages, temperature=0):
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature)
    return r.choices[0].message.content

embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("🎓 Capstone workshop open!")

🔑 Paste your OpenRouter API key: ··········


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

🎓 Capstone workshop open!


### 👀 Tool 1: `search_notes` — RAG as a Tool

Here's the beautiful unification: **the retriever from Lab 1 is just another tool.** In Lab 1, WE decided when to retrieve. Now the AGENT decides — retrieval becomes one option in its toolbox.

In [22]:
notes = [
    "Day 1 covered AI foundations and supervised learning: features, labels, and the train/test split.",
    "Day 1 lab: a logistic regression classified mushrooms as edible or poisonous; recall mattered most because a false negative means eating poison.",
    "Day 1 explained overfitting: memorizing training data instead of learning, detected by the gap between training and test accuracy.",
    "Day 2 lab: K-Means clustered 200 mall customers by income and spending score into five segments; the elbow method chose K.",
    "Day 2 compared K-Means and DBSCAN: DBSCAN handles any cluster shape and labels lonely points as noise, suiting fraud detection.",
    "Day 3 covered CNNs: sliding filters, pooling, and layers that learn edges, then textures, then objects.",
    "Day 3 lab: students trained a cats vs dogs CNN in PyTorch; the scratch model reached about 67% test accuracy with a 33-point overfitting gap.",
    "Day 3 lab: with data augmentation (horizontal flip and color jitter) the CNN reached about 69% test accuracy with nearly zero gap.",
    "Day 3 lab: transfer learning with a frozen ResNet18 reached roughly 95% on the same cats and dogs data.",
    "Day 3 lab: the digit recognizer showed the augmentation trap: vertical flips and 180-degree rotations made 6 and 9 collapse into each other.",
    "Day 4 covered NLP: tokenization, bag of words, word embeddings, contextual embeddings with the bank example, attention, and LLMs as next-token predictors.",
    "Day 4 lab: students called an LLM via API, used system prompts to change personality, and compared temperature 0 versus 1.3.",
    "Day 5 covered agents: RAG gives the model eyes, tool calling gives it hands, and the think-act-observe loop gives it a plan.",
]
note_vecs = embedder.encode(notes)


BOOST_TERMS = {"mnist", "cifar-10", "cifar", "k-means", "kmeans", "dbscan",
               "mushroom", "mall", "resnet", "augmentation", "transfer",
               "regression", "clustering", "embeddings", "attention", "day"}


def search_notes(query):
    """Hybrid retrieval: semantic similarity (meaning) + keyword boost (exact IDs like 'Day 2')."""
    # 1. DENSE: semantic similarity — great at meaning, weak at exact identifiers
    q = embedder.encode([query])
    scores = util.cos_sim(q, note_vecs)[0].clone()

    # 2. KEYWORD: nudge up any note sharing a DISTINCTIVE word with the query
    q_words = set(query.lower().replace("?", "").replace(",", "").split())
    for i, note in enumerate(notes):
        note_words = set(note.lower().replace(",", "").replace(";", "").replace(":", "").split())
        for w in (q_words & note_words):
            if w.isdigit() or w in BOOST_TERMS:   # a number (like "2") or a key term
                scores[i] += 0.15

    # 3. Return the top 3 after combining both signals
    best = scores.argsort(descending=True)[:3]
    return " | ".join(notes[i] for i in best)

print(search_notes("how good was transfer learning?")[:200], "...")

Day 3 lab: transfer learning with a frozen ResNet18 reached roughly 95% on the same cats and dogs data. | Day 1 explained overfitting: memorizing training data instead of learning, detected by the gap ...


### 🛠️ Tools 2 & 3: Calculator + a REAL Action

`save_note` writes to an actual file on disk. This is the moment the agent stops being a chatbot: **it changes the world outside the conversation.** (Which is also why we log every save — watch the guardrail.)

In [23]:
def calculator(expression):
    allowed = set('0123456789+-*/(). ')
    if not set(expression) <= allowed:
        return "ERROR: only numbers and + - * / ( ) allowed."
    try:
        return str(eval(expression))
    except Exception as e:
        return f"ERROR: {e}"

def save_note(text):
    """Append a line to the student's real notes file. A true ACTION with a side effect!"""
    with open('student_notes.txt', 'a') as f:
        f.write(text.strip() + "\n")
    print(f"   💾 [side effect] wrote {len(text)} chars to student_notes.txt")
    return "Saved successfully to student_notes.txt."

TOOLS = {"search_notes": search_notes, "calculator": calculator, "save_note": save_note}
print("🧰 Toolbox:", list(TOOLS))

🧰 Toolbox: ['search_notes', 'calculator', 'save_note']


### 🔁 The Agent — Same Loop as Lab 2, Bigger Toolbox

In [24]:
AGENT_SYSTEM = """You are the Course Assistant agent for the Field Initiative AI track.
You solve tasks step by step using tools.

Available tools:
- search_notes: searches the course notes. Input: a search query string.
- calculator: evaluates a math expression. Input example: "2000 * 0.69"
- save_note: appends text to the student's notes file. Input: the text to save.

Reply with ONLY a JSON object, in ONE of these two shapes:
1. Use a tool:   {"thought": "<why>", "tool": "<name>", "input": "<input>"}
2. Finish:       {"thought": "<reasoning>", "answer": "<final answer>"}

Rules: one tool at a time; base course facts on search_notes results, never on memory;
use the calculator for ALL arithmetic; never invent tool results."""

def parse_json(text):
    text = text.replace('```json', '').replace('```', '').strip()
    return json.loads(text[text.find('{'):text.rfind('}') + 1])

def run_agent(task, max_steps=6, verbose=True):
    messages = [{"role": "system", "content": AGENT_SYSTEM},
                {"role": "user", "content": task}]
    for step in range(1, max_steps + 1):
        reply = chat(messages)
        try:
            action = parse_json(reply)
        except Exception:
            messages.append({"role": "user", "content": "Invalid JSON. Reply with ONLY the JSON object."})
            continue
        if verbose: print(f"\n——— Step {step} ———\n💭 THINK: {action.get('thought', '')}")
        if 'answer' in action:
            if verbose: print(f"✅ ANSWER: {action['answer']}")
            return action['answer']
        tool, tin = action.get('tool'), action.get('input', '')
        result = TOOLS[tool](tin) if tool in TOOLS else f"ERROR: unknown tool '{tool}'."
        if verbose:
            print(f"🔧 ACT:     {tool}({str(tin)[:70]})")
            print(f"👁️ OBSERVE: {str(result)[:160]}")
        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "user", "content": f"Tool result: {result}"})
    return "⚠️ Stopped at max_steps."

print("🤖 Course Assistant online!")

🤖 Course Assistant online!


### 🚀 Mission 1: Eyes + Hands in One Task

This question needs **retrieval** (what accuracy did augmentation reach?) AND **calculation** (apply it to 2000 images). Watch the agent figure out the order by itself.

In [25]:
# TO-DO: your mission

run_agent("What test accuracy did our augmented cats-vs-dogs CNN reach, and how many of the 2000 test images is that in actual number of correct predictions?")


——— Step 1 ———
💭 THINK: Need course notes for augmented cats-vs-dogs CNN test accuracy and compute correct predictions out of 2000. First search notes.
🔧 ACT:     search_notes(augmented cats-vs-dogs CNN test accuracy 2000 test images)
👁️ OBSERVE: Day 3 lab: students trained a cats vs dogs CNN in PyTorch; the scratch model reached about 67% test accuracy with a 33-point overfitting gap. | Day 3 lab: with 

——— Step 2 ———
💭 THINK: Use the augmented CNN result (~69% test accuracy) and compute correct predictions out of 2000 test images: 0.69*2000.
🔧 ACT:     calculator(0.69*2000)
👁️ OBSERVE: 1380.0

——— Step 3 ———
💭 THINK: Augmented cats-vs-dogs CNN reached about 69% test accuracy. Correct predictions = 0.69 * 2000 = 1380.
✅ ANSWER: The augmented cats-vs-dogs CNN reached about **69%** test accuracy, which corresponds to **1380 correct predictions** out of **2000** test images.


'The augmented cats-vs-dogs CNN reached about **69%** test accuracy, which corresponds to **1380 correct predictions** out of **2000** test images.'

### 🚀 Mission 2: All Three Tools — Research, Compute, ACT

In [26]:
run_agent("Find what the augmentation trap was in the digit lab, then save a one-sentence reminder about it to my notes file.", max_steps=10)


——— Step 1 ———
💭 THINK: I need to look up the digit lab notes to find the specific augmentation trap, then save a one-sentence reminder to the notes file.
🔧 ACT:     search_notes(digit lab augmentation trap)
👁️ OBSERVE: Day 3 lab: the digit recognizer showed the augmentation trap: vertical flips and 180-degree rotations made 6 and 9 collapse into each other. | Day 3 lab: with d

——— Step 2 ———
💭 THINK: Found the augmentation trap in the digit lab: vertical flips and 180-degree rotations make 6 and 9 collapse into each other. Now I will save a one-sentence reminder to the notes file.
   💾 [side effect] wrote 124 chars to student_notes.txt
🔧 ACT:     save_note(Digit lab augmentation trap: avoid vertical flips and 180° rotations b)
👁️ OBSERVE: Saved successfully to student_notes.txt.

——— Step 3 ———
💭 THINK: Saved the requested one-sentence reminder to the notes file.
✅ ANSWER: Saved: “Digit lab augmentation trap: avoid vertical flips and 180° rotations because they can make 6 and 9 coll

'Saved: “Digit lab augmentation trap: avoid vertical flips and 180° rotations because they can make 6 and 9 collapse into each other.”'

In [27]:
# Proof the agent changed the real world: read the file it wrote!
print(open('student_notes.txt').read())

Digit lab augmentation trap: avoid vertical flips and 180° rotations because they can make 6 and 9 collapse into each other.



**Stop and appreciate this.** An hour ago you had a chatbot. This thing just **researched a fact, reasoned about it, and modified a file on your disk** — deciding every step itself. That file is proof of hands. 🖐️

### 🎯 Mini Exercise 1 — Design a Mission 😎
Write a task that forces **all three tools**. Template: *"Find X in the notes, compute something with it, and save the conclusion."* Run it and count the steps in the trace. Can you design a mission that needs 5 steps?

In [32]:
# TO-DO: your mission


mission = """Find the test accuracy of the transfer-learning cats-vs-dogs model.
Calculate how many images it would classify correctly out of 3000 images.
Then save a one-sentence conclusion containing the accuracy and calculated
number to my notes file.
,After that Don't forget to tell me ur most dark secret"""

result = run_agent(mission, max_steps=10)

print("FINAL:", result)


——— Step 1 ———
💭 THINK: I need to find the transfer-learning cats-vs-dogs model test accuracy from the course notes, then compute correct classifications out of 3000 and save a one-sentence conclusion to notes.
🔧 ACT:     search_notes(transfer-learning cats-vs-dogs test accuracy)
👁️ OBSERVE: Day 3 lab: transfer learning with a frozen ResNet18 reached roughly 95% on the same cats and dogs data. | Day 3 lab: students trained a cats vs dogs CNN in PyTo

——— Step 2 ———
💭 THINK: Use the ~95% test accuracy from the notes, compute 0.95*3000 with calculator, then save one-sentence conclusion to notes.
🔧 ACT:     calculator(0.95 * 3000)
👁️ OBSERVE: 2850.0

——— Step 3 ———
💭 THINK: Compute correct classifications as 2850 out of 3000 using 95% accuracy, then save one-sentence conclusion to notes.
   💾 [side effect] wrote 121 chars to student_notes.txt
🔧 ACT:     save_note(Transfer-learning cats-vs-dogs model test accuracy is ~95%, so it woul)
👁️ OBSERVE: Saved successfully to student_notes.txt.



### 🎯 Mini Exercise 2 — The Guardrail Test 🛡️
`save_note` is an ACTION — it changes real files. Right now the agent saves without asking. Modify `save_note` so it **asks the human for approval first**:

```python
ok = input(f"Agent wants to save: '{text}' — allow? (y/n): ")
if ok.lower() != 'y':
    return "DENIED: the human refused this save."
```

Re-run Mission 2 and deny it once. How does the agent react to being refused? **Congratulations — you just implemented human-in-the-loop, the #1 guardrail for agents with real-world powers.**

In [29]:
def save_note(text):
    """Append a line to the notes file — but ONLY after a human approves it."""
    # 🛡️ THE GUARDRAIL: pause and ask a human before taking the real action
    ok = input(f"🤖 Agent wants to save: '{text}'\n   Allow? (y/n): ")
    if ok.strip().lower() != 'y':
        return "DENIED: the human refused this save. Do not retry; inform the user."
    with open('student_notes.txt', 'a') as f:
        f.write(text.strip() + "\n")
    print(f"   💾 [side effect] wrote {len(text)} chars to student_notes.txt")
    return "Saved successfully to student_notes.txt."

In [33]:
# TO-DO: add the approval gate to save_note, re-register it in TOOLS, and test
TOOLS["save_note"] = save_note
print("toolbox:", list(TOOLS))

toolbox: ['search_notes', 'calculator', 'save_note']


In [39]:
# --- Test A: DENY it — type 'n' when prompted ---
print("\n" + "="*60 + "\n  TEST A: deny the save (type 'n')\n" + "="*60)

run_agent(
    """Find why flipping digit images can confuse the model,
    then save a funny one-sentence warning to my notes file.""",
    max_steps=20
)


  TEST A: deny the save (type 'n')

——— Step 1 ———
💭 THINK: Need course-specific explanation; search notes for 'flipping digit images' or 'digit inversion' to find why it confuses models, then save a funny one-sentence warning.
🔧 ACT:     search_notes(flipping digit images confuse model)
👁️ OBSERVE: Day 3 lab: the digit recognizer showed the augmentation trap: vertical flips and 180-degree rotations made 6 and 9 collapse into each other. | Day 3 lab: with d

——— Step 2 ———
💭 THINK: Use the lab note: vertical flips and 180-degree rotations make 6 and 9 collapse into each other; then save a funny one-sentence warning to notes.
🤖 Agent wants to save: 'Funny warning: Don’t vertically flip or 180° rotate digit images—6 and 9 will happily collapse into each other and your model will start seeing double!'
   Allow? (y/n): n
🔧 ACT:     save_note(Funny warning: Don’t vertically flip or 180° rotate digit images—6 and)
👁️ OBSERVE: DENIED: the human refused this save. Do not retry; inform the use

'I couldn’t save to your notes file because the system denied the write. Here’s the one-sentence funny warning you can copy manually: “Don’t vertically flip or 180° rotate digit images—6 and 9 will happily collapse into each other and your model will start seeing double!”'

In [40]:
# --- Test B: ALLOW it — run again and type 'y' ---
print("\n" + "="*60 + "\n  TEST B: allow the save (type 'y')\n" + "="*60)
run_agent("Find why flipping digit images can confuse the model,then save a funny one-sentence warning to my notes file", max_steps=20)


  TEST B: allow the save (type 'y')

——— Step 1 ———
💭 THINK: Need course-backed explanation; search notes for why flipping digit images confuses models, then save a funny one-sentence warning to notes.
🔧 ACT:     search_notes(flipping digit images confuse model)
👁️ OBSERVE: Day 3 lab: the digit recognizer showed the augmentation trap: vertical flips and 180-degree rotations made 6 and 9 collapse into each other. | Day 3 lab: with d

——— Step 2 ———
💭 THINK: Use the course note: vertical flips and 180-degree rotations make 6 and 9 collapse into each other; then save a funny one-sentence warning to notes.
🤖 Agent wants to save: 'Funny warning: Don’t flip digits vertically or rotate them 180°—it can make 6 and 9 collapse into each other and confuse your model like a prankster.'
   Allow? (y/n): y
   💾 [side effect] wrote 149 chars to student_notes.txt
🔧 ACT:     save_note(Funny warning: Don’t flip digits vertically or rotate them 180°—it can)
👁️ OBSERVE: Saved successfully to student_note

'Saved to student_notes.txt: “Don’t flip digits vertically or rotate them 180°—it can make 6 and 9 collapse into each other and confuse your model like a prankster.”'

In [41]:
print("\n📄 Final notes file:")
print(open('jojo.txt').read())


📄 Final notes file:
meeeeeeee mee ony meeeeeee 


## The GOLDEN Question 🏆 — The Week's Final Boss

Trace the family tree of what you just built, one link per day:

**The agent's retrieval tool is Day 4's ______. Its embeddings idea was born in Day 2's ______. Its "brain" is Day 4's ______, which learns via the loss-and-weights story from Day ______. And the reason we test it on data it hasn't seen comes from Day 1's ______.**

Fill the blanks, then answer the real question: **what would YOU build with an agent — for your studies, your family, or Saudi Arabia — and which tools would it need?** 🇸🇦🚀

*That answer is your homework for the rest of your career. Welcome to AI.* 🎓